# Midsem Q2: Beam Search Decoding (k=2)

**Vocabulary**: V = {cat, sat, mat, \</s\>}

**Given conditional probabilities across 3 time steps:**

| t=1 | t=2 | t=3 |
|-----|-----|-----|
| P(cat\|\<s\>)=0.5 | P(sat\|cat)=0.6 | P(mat\|cat sat)=0.8 |
| P(sat\|\<s\>)=0.3 | P(mat\|cat)=0.3 | P(\</s\>\|cat sat)=0.2 |
| P(mat\|\<s\>)=0.2 | P(\</s\>\|cat)=0.1 | P(cat\|sat mat)=0.1 |
| | P(cat\|sat)=0.2 | P(\</s\>\|sat mat)=0.9 |
| | P(mat\|sat)=0.7 | |
| | P(\</s\>\|sat)=0.1 | |

In [ ]:
import pandas as pd

print("=" * 70)
print("BEAM SEARCH DECODING WITH BEAM WIDTH k = 2")
print("=" * 70)

# ─────────────────────────────────────────────
# TIME STEP t = 1: Expand from <s>
# ─────────────────────────────────────────────
print("\n" + "─" * 50)
print("TIME STEP t = 1: Expand from <s>")
print("─" * 50)

t1_hypotheses = {
    "cat":  0.5,
    "sat":  0.3,
    "mat":  0.2,
}

print("\nAll expanded hypotheses:")
for hyp, prob in sorted(t1_hypotheses.items(), key=lambda x: -x[1]):
    print(f"  [{hyp}]  P = {prob}")

# Keep top-k=2
t1_top = dict(sorted(t1_hypotheses.items(), key=lambda x: -x[1])[:2])
print(f"\n>>> Top-2 retained: {list(t1_top.keys())} with P = {list(t1_top.values())}")
print(">>> PRUNED: 'mat' (P=0.2) is eliminated.")

# ─────────────────────────────────────────────
# TIME STEP t = 2: Expand from {cat, sat}
# ─────────────────────────────────────────────
print("\n" + "─" * 50)
print("TIME STEP t = 2: Expand from {cat, sat}")
print("─" * 50)

# Expand 'cat' (P=0.5)
print("\nExpanding 'cat' (cumulative P = 0.5):")
expand_cat = {
    "cat sat":   0.5 * 0.6,  # = 0.30
    "cat mat":   0.5 * 0.3,  # = 0.15
    "cat </s>":  0.5 * 0.1,  # = 0.05
}
for hyp, prob in expand_cat.items():
    print(f"  [{hyp}]  P = 0.5 × {prob/0.5:.1f} = {prob:.2f}")

# Expand 'sat' (P=0.3)
print("\nExpanding 'sat' (cumulative P = 0.3):")
expand_sat = {
    "sat cat":   0.3 * 0.2,  # = 0.06
    "sat mat":   0.3 * 0.7,  # = 0.21
    "sat </s>":  0.3 * 0.1,  # = 0.03
}
for hyp, prob in expand_sat.items():
    print(f"  [{hyp}]  P = 0.3 × {prob/0.3:.1f} = {prob:.2f}")

t2_all = {**expand_cat, **expand_sat}
t2_sorted = sorted(t2_all.items(), key=lambda x: -x[1])

print("\nAll 6 expanded hypotheses ranked:")
for rank, (hyp, prob) in enumerate(t2_sorted, 1):
    marker = " ✓ KEPT" if rank <= 2 else " ✗ pruned"
    print(f"  {rank}. [{hyp}]  P = {prob:.2f}{marker}")

t2_top = dict(t2_sorted[:2])
print(f"\n>>> Top-2 retained: {list(t2_top.keys())} with P = {list(t2_top.values())}")

# ─────────────────────────────────────────────
# TIME STEP t = 3: Expand from {cat sat, sat mat}
# ─────────────────────────────────────────────
print("\n" + "─" * 50)
print("TIME STEP t = 3: Expand from {cat sat, sat mat}")
print("─" * 50)

# Expand 'cat sat' (P=0.30)
print("\nExpanding 'cat sat' (cumulative P = 0.30):")
expand_cat_sat = {
    "cat sat mat":   0.30 * 0.8,  # = 0.240
    "cat sat </s>":  0.30 * 0.2,  # = 0.060
}
for hyp, prob in expand_cat_sat.items():
    print(f"  [{hyp}]  P = 0.30 × {prob/0.30:.1f} = {prob:.3f}")

# Expand 'sat mat' (P=0.21)
print("\nExpanding 'sat mat' (cumulative P = 0.21):")
expand_sat_mat = {
    "sat mat cat":   0.21 * 0.1,  # = 0.021
    "sat mat </s>":  0.21 * 0.9,  # = 0.189
}
for hyp, prob in expand_sat_mat.items():
    print(f"  [{hyp}]  P = 0.21 × {prob/0.21:.1f} = {prob:.3f}")

t3_all = {**expand_cat_sat, **expand_sat_mat}
t3_sorted = sorted(t3_all.items(), key=lambda x: -x[1])

print("\nAll 4 expanded hypotheses ranked:")
for rank, (hyp, prob) in enumerate(t3_sorted, 1):
    marker = " ✓ KEPT" if rank <= 2 else " ✗ pruned"
    complete = " [COMPLETE]" if "</s>" in hyp else ""
    print(f"  {rank}. [{hyp}]  P = {prob:.3f}{complete}{marker}")

# ─────────────────────────────────────────────
# FINAL OUTPUT
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("FINAL RESULT")
print("=" * 70)
print("\nTop-2 beams after t=3:")
print("  1. 'cat sat mat'  (P = 0.240) — highest probability")
print("  2. 'sat mat </s>' (P = 0.189) — complete sequence")
print("\n✅ FINAL OUTPUT SEQUENCE: 'cat sat mat'")
print("   Cumulative Probability = 0.5 × 0.6 × 0.8 = 0.240")

### Summary Table

In [ ]:
summary_data = [
    ["t=1", "cat (0.50), sat (0.30), mat (0.20)", "cat (0.50), sat (0.30)"],
    ["t=2", "cat sat (0.30), sat mat (0.21), cat mat (0.15), sat cat (0.06), cat </s> (0.05), sat </s> (0.03)",
     "cat sat (0.30), sat mat (0.21)"],
    ["t=3", "cat sat mat (0.240), sat mat </s> (0.189), cat sat </s> (0.060), sat mat cat (0.021)",
     "cat sat mat (0.240), sat mat </s> (0.189)"],
]

df_summary = pd.DataFrame(summary_data, columns=["Step", "All Expanded Hypotheses (Cumulative P)", "Top-k=2 Retained"])
display(df_summary)

### Why are P(cat|mat) and P(sat|mat) ignored?

At **t=1**, the beam search keeps only the **top-k=2** hypotheses:
- **cat** (P=0.5) ✓
- **sat** (P=0.3) ✓
- **mat** (P=0.2) ✗ **PRUNED**

Since **"mat" was pruned at t=1**, it is **never expanded at t=2**. The conditional probabilities P(cat|mat) and P(sat|mat) are therefore **never computed or used** because "mat" does not exist in the active beam set.

> This is the fundamental trade-off of beam search: by limiting the search space to top-k candidates at each step, it achieves computational efficiency at the cost of potentially missing sequences that pass through low-probability intermediate states. In this case, any sequence starting with "mat" (e.g., "mat cat ...", "mat sat ...") is permanently eliminated regardless of how high the subsequent conditional probabilities might be.